In [ ]:
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss

torch.cuda.empty_cache()

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
embedding_model_name = "all-MiniLM-L6-v2"
digits = 2

/home/panyi/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [3]:
import pandas as pd
reviews = pd.read_pickle("data/data_processed/review_clean.pkl")
# reviews

In [ ]:
import re, json

def extract(text) :
    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        data = json.loads(match.group(0))
    return data

## Without RAG

In [ ]:
def prompt_generator(context, content, category, digit: int = 2):

    if content is not None:
        content = content[:50000] if len(content) > 50000 else content
    category = eval(category)
    
    prompt = f"""Context: {context}
Please classify the business using the North American Industry Classification System (NAICS) 
based on the following content and category information provided by the business. 
Provide the top-3 most probable {digit}-digit codes. Use the context above if it is helpful.

Business content: {content}
Business category: {category}

Please parse "answer" and output them in JSON format.

EXAMPLE JSON OUTPUT:
{{
    "First NAICS code": "CODE 1",
    "Second NAICS code": "CODE 2",
    "Third NAICS code": "CODE 3"
}}
"""
    
    return prompt

def get_llm_response(context, content, category, digit: int = 2, max_len: int = 2048, temperature = 0.0001) :
    
    def post_processing(generated_token_ids) :
        return generated_token_ids[:, len(model_inputs["input_ids"][0]):]

    prompt = prompt_generator(context, content, category, digit)
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant for insurance companies. Please give short and straightforward answers."},
        {"role": "user", "content": prompt}
    ]
    
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([input_text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_len,
        pad_token_id=tokenizer.eos_token_id,
        temperature=temperature,
    )
    generated_token_ids_post_processed = post_processing(generated_ids)
    
    response = tokenizer.batch_decode(generated_token_ids_post_processed, skip_special_tokens=True)[0]
    
    return response

In [ ]:
# responses = list(pd.read_csv("response_{}.csv".format(digits)).values.flatten())
responses = []
for content, category in tqdm(zip(reviews["content"], reviews["categories"]), total = len(reviews["content"])) :
    response = get_llm_response("", content, category, digits)
    try :
        response = extract(response)
    except :
        print(response)
        pass
    responses.append(response)
    pd.DataFrame([str(item) for item in responses]).to_csv("response_{}.csv".format(digits), index = False)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Okay, so I need to figure out the NAICS codes for the business provided. Let me start by reading through the content carefully. The business is called Cabot Corporation, and it's a global specialty chemicals and performance materials company. They invest in newsroom products and applications, which suggests they're involved in research and development. They also provide sustainability information, which is a key area in NAICS.

Looking at the NAICS codes, I remember that the Chemical Industry is a major category, and within that, there are specific codes for specialty chemicals and performance materials. Cabot is a leading company in this sector, so I think the primary code would be 3101.00, which is for specialty chemicals and performance materials.

They also mention sustainability, which falls under the Industry category. The specific code for sustainability in the Chemical Industry is 3101.01. Additionally, since they're involved in business service and wholesale, the code 3101.02 

## With RAG

In [31]:
# extract documents and convert to JSON lines format
docs = pd.read_pickle("data/data_processed/naics_clean_{}.pkl".format(digits))
docs = docs[["naics", "title", "description"]].to_json(orient ='records', lines=True).splitlines()
docs = [eval(doc) for doc in docs]

In [ ]:
# prepare embedding model
embedding_model = SentenceTransformer(embedding_model_name, model_kwargs = {"dtype": torch.float16})

# combine title and description for embedding
embed_doc = []
for doc in docs:
    embed_doc.append(f"{doc['title']} {doc['description']}")
embed_doc = embedding_model.encode(embed_doc, show_progress_bar=True, convert_to_tensor=True)

# create FAISS index
index = faiss.IndexFlatL2(embed_doc.shape[1])
index.add(embed_doc.cpu().numpy())
faiss.write_index(index, "data/data_processed/naics_index_{}.faiss".format(digits))

# put embedding model to cpu to save GPU memory
embedding_model.cpu()

def get_rag_llm_response(content, category, top_k: int = 5, digit: int = 2, max_len: int = 1024, max_ind_context: int = 50000, temperature = 0.0001):

    # embed the content and category
    query = embedding_model.encode([f"{content} {category}"], convert_to_tensor=True)#.cpu().numpy()
    # search the index
    D, I = index.search(query, top_k)
    
    # retrieve the top-k documents
    context = ""
    for i, idx in enumerate(I[0]):
        case = docs[idx]
        tmp_context = f"NAICS code :{case['naics']}. Title: {case['title']}. Description: {case['description']}\n"
        context += (tmp_context[:max_ind_context] if len(tmp_context) > max_ind_context else tmp_context)
    # get the response from LLM
    response = get_llm_response(context, content, category, digit, max_len, temperature)
    
    return response

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.16it/s]


In [ ]:
# responses = list(pd.read_csv("response_rag_{}.csv".format(digits)).values.flatten())
responses = []
for content, category in tqdm(zip(reviews["content"], reviews["categories"]), total = len(reviews["content"])) :
    response = get_rag_llm_response(content, category, digit=digits)
    try :
        response = extract(response)
    except :
        pass
    responses.append(response)
    pd.DataFrame([str(item) for item in responses]).to_csv("response_rag_{}.csv".format(digits), index = False)

Token indices sequence length is longer than the specified maximum sequence length for this model (24179 > 16384). Running this sequence through the model will result in indexing errors
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Okay, so I need to figure out the NAICS category for the business mentioned in the text. Let me start by reading through the text carefully. 

The text mentions several things: cabot corporation, their products, their commitment to sustainability, and their operations. They talk about their products, their technology, their environmental impact, and their people. 

Looking at the text, I see that they mention their products, their technology, their environmental impact, and their people. They also talk about their commitment to sustainability and their operations. 

I remember that the text mentions their products, their technology, their environmental impact, and their people. They also talk about their commitment to sustainability and their operations. 

I need to find the NAICS category that corresponds to these elements. Let me think about the text. 

The text mentions their products, their technology, their environmental impact, and their people. They also talk about their commitm

## Evaluation

In [1]:
import numpy as np
import pandas as pd
import itertools
review_clean_path = "data/data_processed/review_clean.pkl"
review_clean = pd.read_pickle(review_clean_path)
digits = 2

In [ ]:
import numpy as np
na_types = [None, "novalue", "None", np.nan, "nan", "none"]

def recode_naics(naics, digits: int = 2):
    """
    Recode the NAICS codes to 2/4/6 digits.
    """
    if np.isnan(naics) or naics in na_types:
        return naics
    elif digits == 2:
        new_naics = int(naics // 10000)
        if new_naics >= 31 and new_naics <= 33:
            return "31-33"
        elif new_naics >= 44 and new_naics <= 45:
            return "44-45"
        elif new_naics >= 48 and new_naics <= 49:
            return "48-49"
        else:
            return str(new_naics)
    elif digits == 4:
        return str(naics // 100)
    elif digits == 6:
        return str(naics)
    
def process_row(row) :
    try :
        row = eval(row).values()
        if digits == 2:
            row = [recode_naics(int(code) * 10000, digits) if int(code) < 100 else "00" for code in row]
        elif digits == 4:
            row = [recode_naics(int(code) * 100, digits) if int(code) < 10000 else "0000" for code in row]
        elif digits == 6:
            row = [recode_naics(int(code), digits) for code in row]
        return row
    except:
        if digits == 2:
            return ["00", "00", "00"]
        elif digits == 4:
            return ["0000", "0000", "0000"]
        elif digits == 6:
            return ["000000", "000000", "000000"]

def get_accuracy(review_clean, label_cols: list = ["naics1"], verbose: bool = False):

    # top-k accuracy
    # if intersection of the predicted and actual labels is not empty, then the prediction is correct
    def _accuracy(pred, actual):
        return len(set(pred) & set(actual)) > 0

    # calculate the accuracy of the predictions
    # get top-1 prediction accuracy
    preds = review_clean["naics_pred"].apply(lambda x: x[0])
    # merge all label columns into a single list column
    review_clean["naics_recode"] = review_clean[label_cols].values.tolist()
    # remove empty labels & convert each to integer
    review_clean["naics_recode"] = review_clean["naics_recode"].apply(
        lambda x: [int(i) for i in x if str(i) != "nan"]
    )
    # recode the labels to 2/4/6 digits
    review_clean["naics_recode"] = review_clean["naics_recode"].apply(
        lambda x: [recode_naics(_x, digits) for _x in x]
    )
    accuracy_1 = sum(
        [pred in _class for _class, pred in zip(review_clean["naics_recode"], preds)]
    ) / len(review_clean)

    # get top-2 prediction accuracy
    preds = review_clean["naics_pred"].apply(lambda x: x[:2])
    accuracy_2 = sum(
        [
            _accuracy(pred, _class)
            for _class, pred in zip(review_clean["naics_recode"], preds)
        ]
    ) / len(review_clean)

    # get top-3 prediction accuracy
    preds = review_clean["naics_pred"]
    accuracy_3 = sum(
        [
            _accuracy(pred, _class)
            for _class, pred in zip(review_clean["naics_recode"], preds)
        ]
    ) / len(review_clean)

    if verbose:
        print("Top-1 prediction accuracy: {:.2f}%".format(accuracy_1 * 100))
        print("Top-2 prediction accuracy: {:.2f}%".format(accuracy_2 * 100))
        print("Top-3 prediction accuracy: {:.2f}%".format(accuracy_3 * 100))

    return accuracy_1, accuracy_2, accuracy_3

In [3]:
# regular llm
response = pd.read_csv("response_{}.csv".format(digits))
review_clean["naics_pred"] = response["0"].apply(process_row)
unique_codes = list(review_clean["naics_pred"].values.flatten())
unique_codes = np.unique(list(itertools.chain(*unique_codes)))
get_accuracy(review_clean, ["naics_code_1_mapped_2022", "naics_code_2_mapped_2022"])

[True, True, False, False, True, False, False, False, False, True, False, False, True, True, True, True, True, False, True, False, True, False, True, False, False, True, False, True, False, False, False, False, False, False, False, False, False, False, False, True, False, True, False, True, True, False, False, False, False, True, False, False, True, True, False, True, False, True, False, False, False, False, True, True, False, False, False, True, False, False, False, False, False, True, False, False, False, False, True, False, False, True, False, True, False, True, False, True, True, True, False, False, True, False, False, True, True, True, False, False, True, False, False, True, True, True, False, True, False, False, False, False, False, True, False, False, False, False, True, False, False, False, True, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, True, True, True, True, True, True, True, False, True, False, False, True

(0.2306070826306914, 0.2765598650927487, 0.30269814502529513)

In [4]:
# rag llm
response_rag = pd.read_csv("response_rag_{}.csv".format(digits))
review_clean["naics_pred"] = response_rag["0"].apply(process_row)
unique_codes = list(review_clean["naics_pred"].values.flatten())
unique_codes = np.unique(list(itertools.chain(*unique_codes)))
get_accuracy(review_clean, ["naics_code_1_mapped_2022", "naics_code_2_mapped_2022"])

[False, False, True, False, True, True, False, False, True, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, True, False, False, False, False, True, False, True, False, True, False, True, False, True, False, False, False, False, False, False, False, False, True, True, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, True, False, False, False, True, False, False, True, False, False, True, True, False, True, True, False, False, False, False, False, True, True, False, False, False, False, False, True, False, False, True, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, True, False, False, False, False, True, False, True, Tr

(0.17158516020236086, 0.21205733558178752, 0.224283305227656)